In [ ]:
from pathlib import Path
import os

root = Path(os.getcwd()).resolve().parents[1]  # вверх на 2 уровня
!python {str(root / "scripts" / "bootstrap.py")}

%load_ext autoreload
%autoreload 2

from hydra import initialize_config_dir, compose
from hydra.utils import instantiate

config_name = "default"

with initialize_config_dir(config_dir=str(root / "configs"), version_base="1.3"):
    cfg = compose(config_name=config_name)

In [ ]:
import torch, os
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.data import Batch

from geomml.models import build as build_model
from geomml.datasets import build as build_dataset
from geomml.losses.kendall import * 
from geomml.utils.loader import build as build_loaders
from geomml.utils.plot_history import *
from geomml.trainers.multi_task import MultiTaskTrainer
from geomml.trainers.evaluate_model import *

DATA = "data"
dataset  = torch.load(os.path.join(DATA, "qm9s.pt"), map_location="cpu")

print(type(dataset ))

print(type(dataset))
print(len(dataset))

sample = dataset[0]

print(type(sample))
print(sample)

del dataset

1. GeomML-признаки:
* SOAP (DScribe), 
* ACSF, 
* MBTR, 
* Coulomb Matrix 
* эмбеддинги уже существующей модели GeomML?

2. TDA:
* Persistence Diagram;
* Persistence Image;
* Persistence Landscape;
* Betti Curve.

In [ ]:
src=os.path.join(DATA,"qm9s.pt")
out=os.path.join(DATA,"qm9s_processed.pt")

if not os.path.exists(out):
    !python {str(root / "scripts" / "preprocess_qm9s.py")} {src} {out}
else:
    print(f"Already exists: {out}")

dataset=build_dataset(
    name="qm9s",
    root=os.path.join(DATA, "qm9s_processed.pt"),
    normalize=True
)
loaders=build_loaders( dataset, batch_size=64 )

# Проверка перед запуском обучения

In [ ]:
batch=next(iter(loaders.train))
model=build_model(name="qm9s_dimenet")
out=model(batch)

print(out["dipole"].shape)
print(out["polar"].shape)

In [ ]:
data_qm9s_processed=torch.load("data/qm9s_processed.pt", map_location="cpu")

print(data_qm9s_processed[0].dipole.shape)
print(data_qm9s_processed[0].polar.shape)

In [ ]:
from torch_geometric.data import Batch

b=Batch.from_data_list(
    data_qm9s_processed[:64]
)

print(type(b))
print(b.dipole.shape)
print(b.polar.shape)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch=next(iter(loaders.train))

print(type(batch))
print(type(batch.dipole))
print(batch.dipole.shape)
print(batch.polar.shape)
print(type(data_qm9s_processed[0]))

batch=batch.to(device)
model=model.to(device)
out=model(batch)
loss=model.loss_fn(out,batch)

print(out["dipole"].shape)
print(out["polar"].shape)
print(loss)

In [ ]:
batch=next(iter(loaders.train))

print(batch.dipole.shape)
print(batch.polar.shape)

del loaders, out, model, batch, data_qm9s_processed


In [ ]:
loaders=build_loaders( dataset, batch_size=32, train_size=0.8, valid_size=0.1)
model=build_model( "qm9s_dimenet", hidden_dim=256, layers=4 )
optimizer=instantiate( cfg.optimizer, params=model.parameters() )
scheduler=instantiate( cfg.scheduler, optimizer )

N_EPOCHS=2

model,history=MultiTaskTrainer(
    model=model,
    optimizer=optimizer,
    scheduler=scheduler,
    patience=20,
    min_delta=1e-4,
    checkpoint_name="best_qm9s_dimenet",
).fit(
    train_loader=loaders.train,
    valid_loader=loaders.valid,
    num_epochs=N_EPOCHS
)

plot_history(history, title="[qm9s_dimenet] model training [default.loss] curves on [QM9s] dataset")
print( evaluate_qm9s( model, loaders.test, device ) )

del model, loaders, optimizer, history, scheduler